# Training YOLO26 (2-class)

Dataset: `dataset_split` dengan `data.yaml` (kelas: normal, anomali).
Catatan: set `MODEL_PATH` ke bobot atau file YAML YOLO26 yang Anda pakai.


In [1]:
from pathlib import Path
from collections import Counter
from PIL import Image

base = Path('dataset_split')
sets = ['train', 'valid', 'test']
for s in sets:
    img_dir = base / s / 'images'
    lab_dir = base / s / 'labels'
    img_paths = list(img_dir.glob('*.*'))
    sizes = Counter()
    for p in img_paths:
        with Image.open(p) as im:
            sizes[im.size] += 1
    class_counts = Counter()
    for p in lab_dir.glob('*.txt'):
        text = p.read_text(encoding='utf-8').strip()
        if not text:
            continue
        for line in text.splitlines():
            parts = line.split()
            if len(parts) < 5:
                continue
            cls = int(float(parts[0]))
            class_counts[cls] += 1
    print(s, 'images', len(img_paths), 'unique_sizes', len(sizes), 'top_sizes', sizes.most_common(3))
    print(s, 'class_counts', dict(class_counts))


train images 993 unique_sizes 1 top_sizes [((640, 360), 993)]
train class_counts {0: 693, 1: 301}
valid images 286 unique_sizes 1 top_sizes [((640, 360), 286)]
valid class_counts {0: 199, 1: 87}
test images 144 unique_sizes 1 top_sizes [((640, 360), 144)]
test class_counts {0: 100, 1: 45}


In [2]:
import importlib.util
import sys
import subprocess

if importlib.util.find_spec('ultralytics') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-U', 'ultralytics'])


In [1]:
import torch

print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

2.12.0.dev20260408+cu128
12.8
True
NVIDIA GeForce RTX 5060 Laptop GPU


In [2]:
from ultralytics import YOLO
import torch

MODEL_PATH = 'yolo26n.pt'
DATA_YAML = 'data.yaml'
IMG_SIZE = 640
EPOCHS = 100
BATCH = 16
DEVICE = 0 if torch.cuda.is_available() else 'cpu'

model = YOLO(MODEL_PATH)
results = model.train(
    data=DATA_YAML,
    imgsz=IMG_SIZE,
    epochs=EPOCHS,
    batch=BATCH,
    device=DEVICE,
    project='runs/train',
    name='yolo26_lele'
)


New https://pypi.org/project/ultralytics/8.4.51 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.48  Python-3.10.20 torch-2.12.0.dev20260408+cu128 CUDA:0 (NVIDIA GeForce RTX 5060 Laptop GPU, 8151MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scal

In [1]:
!yolo export model=runs/detect/runs/train/yolo26_lele-2/weights/best.pt format=ncnn

Ultralytics 8.4.51  Python-3.10.20 torch-2.12.0.dev20260408+cu128 CPU (Intel Core i7-14700HX)
WARNING NCNN export does not support end2end models, disabling end2end branch.
YOLO26n summary (fused): 146 layers, 2,495,084 parameters, 0 gradients, 5.2 GFLOPs

PyTorch: starting from 'runs\detect\runs\train\yolo26_lele-2\weights\best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 6, 8400) (5.1 MB)
requirements: Ultralytics requirement ['ncnn'] not found, attempting AutoUpdate...
Using Python 3.10.20 environment at: C:\Users\TROYY\anaconda3\envs\skripkir
Resolved 1 package in 410ms
 Downloaded ncnn
Prepared 1 package in 2.40s
Installed 1 package in 11ms
 + ncnn==1.0.20260114

requirements: AutoUpdate success  3.7s
WARNING requirements: Restart runtime or rerun command for updates to take effect

requirements: Ultralytics requirement ['pnnx'] not found, attempting AutoUpdate...
Using Python 3.10.20 environment at: C:\Users\TROYY\anaconda3\envs\skripkir
Resolved 11 packages

pnnxparam = runs/detect/runs/train/yolo26_lele-2/weights/best_ncnn_model/model.pnnx.param
pnnxbin = runs/detect/runs/train/yolo26_lele-2/weights/best_ncnn_model/model.pnnx.bin
pnnxpy = runs/detect/runs/train/yolo26_lele-2/weights/best_ncnn_model/model_pnnx.py
pnnxonnx = runs/detect/runs/train/yolo26_lele-2/weights/best_ncnn_model/model.pnnx.onnx
ncnnparam = runs/detect/runs/train/yolo26_lele-2/weights/best_ncnn_model/model.ncnn.param
ncnnbin = runs/detect/runs/train/yolo26_lele-2/weights/best_ncnn_model/model.ncnn.bin
ncnnpy = runs/detect/runs/train/yolo26_lele-2/weights/best_ncnn_model/model_ncnn.py
fp16 = 0
optlevel = 2
device = cpu
inputshape = [1,3,640,640]f32
inputshape2 = 
customop = 
moduleop = 
get inputshape from traced inputs
inputshape = [1,3,640,640]f32
############# pass_level0
inline module = torch.nn.modules.linear.Identity
inline module = ultralytics.nn.modules.block.Attention
inline module = ultralytics.nn.modules.block.Bottleneck
inline module = ultralytics.nn.modules